In [ ]:
import pandas as pd
import numpy as np
import math
import sqlite3

In [129]:
# Create the dataframe to hold the OBP at each count
count_table_cols = ['balls','strikes','obp']

# Data taken from baseball reference
# OBP is the from that count and onward in the at bat, not just the next pitch
count_df = pd.DataFrame(columns=count_table_cols)
count_df = pd.DataFrame([
    {'balls': 0, 'strikes': 0, 'obp': 0.345},
    {'balls': 0, 'strikes': 1, 'obp': 0.263},
    {'balls': 0, 'strikes': 2, 'obp': 0.197},
    {'balls': 0, 'strikes': 3, 'obp': 0.000},  # strikeout
    {'balls': 1, 'strikes': 0, 'obp': 0.384},
    {'balls': 1, 'strikes': 1, 'obp': 0.307},
    {'balls': 1, 'strikes': 2, 'obp': 0.224},
    {'balls': 1, 'strikes': 3, 'obp': 0.000},  # strikeout
    {'balls': 2, 'strikes': 0, 'obp': 0.509},
    {'balls': 2, 'strikes': 1, 'obp': 0.400},
    {'balls': 2, 'strikes': 2, 'obp': 0.294},
    {'balls': 2, 'strikes': 3, 'obp': 0.000},  # strikeout
    {'balls': 3, 'strikes': 0, 'obp': 0.757},
    {'balls': 3, 'strikes': 1, 'obp': 0.610},
    {'balls': 3, 'strikes': 2, 'obp': 0.460},
    {'balls': 3, 'strikes': 3, 'obp': 0.000},  # strikeout
    {'balls': 4, 'strikes': 0, 'obp': 1.000},  # walk
    {'balls': 4, 'strikes': 1, 'obp': 1.000},  # walk
    {'balls': 4, 'strikes': 2, 'obp': 1.000},  # walk
], columns=count_table_cols)

In [130]:
count_delta_cols = ['balls','strikes','obp_delta']
count_delta_df = pd.DataFrame(columns=count_delta_cols)
count_delta_df = pd.DataFrame([
    {'balls': 0, 'strikes': 0},
    {'balls': 0, 'strikes': 1},
    {'balls': 0, 'strikes': 2},
    {'balls': 1, 'strikes': 0},
    {'balls': 1, 'strikes': 1},
    {'balls': 1, 'strikes': 2},
    {'balls': 2, 'strikes': 0},
    {'balls': 2, 'strikes': 1},
    {'balls': 2, 'strikes': 2},
    {'balls': 3, 'strikes': 0},
    {'balls': 3, 'strikes': 1},
    {'balls': 3, 'strikes': 2},
], columns=count_delta_cols)

# Calculate the change in OBP for a strike vs a ball
# aka how much of an OBP increase (on average) will a batter see if in the following count if they correctly challenge a called strike in that count
for index, row in count_delta_df.iterrows():
    balls = row['balls']
    strikes = row['strikes']
    
    # Get the OBP for the pitch is a strike (note that if there are already 2 strikes, the OBP for a strike is 0.000 since the batter would strikeout)
    strike_obp = count_df[(count_df['balls'] == balls) & (count_df['strikes'] == strikes + 1)]['obp'].values[0] if strikes < 2 else 0.0
    # And for a ball (note that if there are already 3 balls, the OBP for a ball is 1.000 since the batter would walk)
    ball_obp = count_df[(count_df['balls'] == balls + 1) & (count_df['strikes'] == strikes)]['obp'].values[0] if balls < 3 else 1.0

    obp_delta = abs(strike_obp - ball_obp)
    count_delta_df.at[index, 'obp_delta'] = obp_delta

        

count_delta_df.head(12)


,balls,strikes,obp_delta
0,0,0,0.121
1,0,1,0.110
2,0,2,0.224
3,1,0,0.202
4,1,1,0.176
5,1,2,0.294
6,2,0,0.357
7,2,1,0.316
8,2,2,0.460
9,3,0,0.390


In [131]:
# Add count deltas to database
conn = sqlite3.connect('../data/processed/abs_challenge.db')
count_delta_df.to_sql('count_deltas', conn, if_exists='replace', index=False)

testQuery = "SELECT * FROM count_deltas"
test_df = pd.read_sql_query(testQuery, conn)
test_df.head()

,balls,strikes,obp_delta
0,0,0,0.121
1,0,1,0.110
2,0,2,0.224
3,1,0,0.202
4,1,1,0.176


In [132]:
# Make the ABS Challenges database connection
conn = sqlite3.connect('../data/processed/abs_challenge.db')

challenge_results_cols = ['player_name', 'batter', 'balls', 'strikes', 'description']
challenge_results_df = pd.read_csv(
    '../data/raw/savant_data_abs_challenge.csv',
    na_values='\\N',
    usecols=challenge_results_cols,
    low_memory=False
)

challenge_results_df.to_sql('batter_challenge_results', conn, if_exists='replace', index=False)

conn.execute("""
ALTER TABLE batter_challenge_results
RENAME COLUMN batter TO batter_id;
""")
conn.commit()

TestQuery = "SELECT * FROM batter_challenge_results;"
test_df = pd.read_sql_query(TestQuery, conn)
print(test_df.head())


         player_name  batter_id    description  balls  strikes
0      Abreu, Wilyer     677800           ball      3        2
1  Rafaela, Ceddanne     678882  called_strike      3        2
2      Bregman, Alex     608324  called_strike      1        2
3      Herrera, Iván     671056           ball      0        0
4   Bogaerts, Xander     593428  called_strike      0        2


In [133]:
challenge_leaders_query = """SELECT player_name, batter_id, COUNT(*) as challenge_count
FROM batter_challenge_results
GROUP BY batter_id
ORDER BY challenge_count DESC
LIMIT 10;"""

challenge_leaders_df = pd.read_sql_query(challenge_leaders_query, conn)
challenge_leaders_df.head(10)

,player_name,batter_id,challenge_count
0,"Stewart, Sal",701398,30
1,"Sánchez, Gary",596142,19
2,"McGonigle, Kevin",805808,17
3,"Smith, Cam",701358,17
4,"Caballero, José",676609,17
5,"Garcia, Maikel",672580,17
6,"Harper, Bryce",547180,17
7,"Schanuel, Nolan",694384,16
8,"Martin, Austin",668885,16
9,"Acuña Jr., Ronald",660670,16


In [134]:
# Add Challenge Value to database
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(batter_challenge_results);")
existing_cols = {row[1] for row in cursor.fetchall()}

if "challenge_value" not in existing_cols:
    cursor.execute("ALTER TABLE batter_challenge_results ADD COLUMN challenge_value INTEGER DEFAULT NULL;")


conn.commit()

In [135]:
# Challenge Value Calculation
# Correct challenge (pitch ruled ball after review): +1
# Incorrect challenge (pitch ruled strike after review): -1
# Count Quality:  + the obs delta for the count of the pitch being challenged

# ie: 
# correct challenge on a 0-0 count: 1.018
# incorrect challenge on a 2-0 count: -0.389

# The higher the value, the more 'worth it' the challenge was for the batter, in terms of how much it increased their chances of reaching base on that plate appearance

add_challenge_value_query = """UPDATE batter_challenge_results
SET challenge_value = CASE
    WHEN description = 'ball' THEN 1 + (
        SELECT obp_delta FROM count_deltas
        WHERE count_deltas.balls = batter_challenge_results.balls
        AND count_deltas.strikes = batter_challenge_results.strikes
    )
    WHEN description = 'called_strike' THEN -1 + (
        SELECT obp_delta FROM count_deltas
        WHERE count_deltas.balls = batter_challenge_results.balls
        AND count_deltas.strikes = batter_challenge_results.strikes
    )
END;"""

conn.execute(add_challenge_value_query)
conn.commit()

In [136]:
# Check that it worked
test_query = """SELECT *
FROM batter_challenge_results
LIMIT 10;"""
test_df = pd.read_sql_query(test_query, conn)
test_df.head(10)

,player_name,batter_id,description,balls,strikes,challenge_value
0,"Abreu, Wilyer",677800,ball,3,2,2.000
1,"Rafaela, Ceddanne",678882,called_strike,3,2,0.000
2,"Bregman, Alex",608324,called_strike,1,2,-0.706
3,"Herrera, Iván",671056,ball,0,0,1.121
4,"Bogaerts, Xander",593428,called_strike,0,2,-0.776
5,"Soderstrom, Tyler",691016,ball,1,1,1.176
6,"Langford, Wyatt",694671,called_strike,2,2,-0.540
7,"Walker, Christian",572233,called_strike,2,0,-0.643
8,"Lee, Hao-Yu",701678,ball,0,0,1.121
9,"Loftin, Nick",679845,ball,0,0,1.121


In [137]:
# Make the ABS Challenges database connection
conn = sqlite3.connect('../data/processed/abs_challenge.db')

challenge_results_cols = ['player_name', 'fielder_2', 'balls', 'strikes', 'description']
challenge_results_df = pd.read_csv(
    '../data/raw/savant_data_abs_challenge_catchers.csv',
    na_values='\\N',
    usecols=challenge_results_cols,
    low_memory=False
)

challenge_results_df.to_sql('catcher_challenge_results', conn, if_exists='replace', index=False)

conn.execute("""
ALTER TABLE catcher_challenge_results
RENAME COLUMN fielder_2 TO catcher_id;
""")
conn.commit()

TestQuery = "SELECT * FROM catcher_challenge_results;"
test_df = pd.read_sql_query(TestQuery, conn)
test_df.head()

,player_name,description,balls,strikes,catcher_id
0,"Jackson, Alex",called_strike,2,2,656577
1,"Herrera, Iván",called_strike,3,2,671056
2,"Fortes, Nick",called_strike,0,0,663743
3,"Realmuto, J.T.",called_strike,1,2,592663
4,"Langeliers, Shea",called_strike,0,1,669127


In [138]:
# Add Challenge Value to database
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(catcher_challenge_results);")
existing_cols = {row[1] for row in cursor.fetchall()}

if "challenge_value" not in existing_cols:
    cursor.execute("ALTER TABLE catcher_challenge_results ADD COLUMN challenge_value INTEGER DEFAULT NULL;")


conn.commit()

In [139]:
# Challenge Value Calculation
# Correct challenge (pitch ruled strike after review): +1
# Incorrect challenge (pitch ruled ball after review): -1
# Count Quality:  + the obs delta for the count of the pitch being challenged

# ie: 
# correct challenge on a 0-0 count: 1.018
# incorrect challenge on a 2-0 count: -0.389

# The higher the value, the more 'worth it' the challenge was for the hitter in terms of how much it decreased the batters chances of reaching base on that plate appearance 
add_challenge_value_query = """UPDATE catcher_challenge_results
SET challenge_value = CASE
    WHEN description = 'called_strike' THEN 1 + (
        SELECT obp_delta FROM count_deltas
        WHERE count_deltas.balls = catcher_challenge_results.balls
        AND count_deltas.strikes = catcher_challenge_results.strikes
    )
    WHEN description = 'ball' THEN -1 + (
        SELECT obp_delta FROM count_deltas
        WHERE count_deltas.balls = catcher_challenge_results.balls
        AND count_deltas.strikes = catcher_challenge_results.strikes
    )
END;"""

conn.execute(add_challenge_value_query)
conn.commit()

In [140]:
challenge_leaders_query = """SELECT player_name, catcher_id, COUNT(*) as challenge_count
FROM catcher_challenge_results
GROUP BY catcher_id
ORDER BY challenge_count DESC
LIMIT 10;"""

challenge_leaders_df = pd.read_sql_query(challenge_leaders_query, conn)
challenge_leaders_df.head(10)

,player_name,catcher_id,challenge_count
0,"Contreras, William",661388,90
1,"Quero, Edgar",700337,68
2,"Langeliers, Shea",669127,61
3,"Fortes, Nick",663743,60
4,"Stephenson, Tyler",663886,59
5,"O'Hoppe, Logan",681351,57
6,"Bailey, Patrick",672275,56
7,"Goodman, Hunter",696100,55
8,"Caratini, Victor",605170,52
9,"Perez, Salvador",521692,50


In [141]:
conn.close()